# `steady_state.py` — Test Suite Demo

This notebook re-runs all eight unit tests from `tests/test_steady_state.py` and
produces plots to visualise the data used in each test.

Tests covered:

| # | Test | Function |
|---|------|----------|
| 1 | Simple diagonal A-matrix concentrations | `_steady_state_concentrations_from_dataset` |
| 2 | No A-matrix → empty dict | `_steady_state_concentrations_from_dataset` |
| 3 | End-to-end values | `compute_steady_state_spectra` |
| 4 | Correct output dimensions | `compute_steady_state_spectra` |
| 5 | Skips dataset without A-matrix | `compute_steady_state_spectra` |
| 6 | Multi-megacomplex accumulation | `compute_steady_state_spectra` |
| 7 | `exclude_species` removes species | `compute_steady_state_spectra` |
| 8 | Excluding absent species is a no-op | `compute_steady_state_spectra` |

## Setup — imports and shared helpers

In [ ]:
from __future__ import annotations

import importlib

import matplotlib.pyplot as plt
import numpy as np
import pygta_local_extras.analysis.steady_state as _ss_mod
import xarray as xr

importlib.reload(_ss_mod)

from pygta_local_extras.analysis.steady_state import (
    _steady_state_concentrations_from_dataset,
    compute_steady_state_spectra,
)

print("Module loaded:", _ss_mod.__file__)

In [ ]:
# ── Shared helper functions (mirrors the test file) ──────────────────────────


def _make_dataset(
    *,
    n_spectral: int = 5,
    species: list[str],
    rates: np.ndarray,
    a_matrix: np.ndarray,
    sas: np.ndarray,
    mc_label: str = "mc1",
) -> xr.Dataset:
    """Build a minimal xr.Dataset that looks like a pyglotaran result dataset."""
    spectral = np.linspace(600, 750, n_spectral)
    lifetimes = 1.0 / rates
    component_name = f"component_{mc_label}"
    species_name = f"species_{mc_label}"

    a_mat_da = xr.DataArray(
        a_matrix,
        dims=(component_name, species_name),
        coords={
            component_name: np.arange(1, rates.size + 1),
            f"rate_{mc_label}": (component_name, rates),
            f"lifetime_{mc_label}": (component_name, lifetimes),
            species_name: species,
        },
    )
    sas_da = xr.DataArray(
        sas,
        dims=("spectral", "species"),
        coords={"spectral": spectral, "species": species},
    )
    return xr.Dataset(
        {
            f"a_matrix_{mc_label}": a_mat_da,
            "species_associated_spectra": sas_da,
        }
    )


class _FakeResult:
    def __init__(self, data: dict[str, xr.Dataset]) -> None:
        self.data = data


def _assert(condition: bool, msg: str = "") -> None:
    assert condition, msg
    print(f"  PASS  {msg}" if msg else "  PASS")


print("Helpers ready.")

---
## Test 1 — Simple diagonal A-matrix

`_steady_state_concentrations_from_dataset`: for a diagonal A-matrix,
$c_{SS,i} = A_{ii} \cdot \tau_i$.

In [ ]:
species_t1 = ["s1", "s2"]
rates_t1 = np.array([1.0, 2.0])
a_matrix_t1 = np.diag([0.5, 0.8])  # (2, 2)
sas_t1 = np.ones((5, 2))

ds_t1 = _make_dataset(species=species_t1, rates=rates_t1, a_matrix=a_matrix_t1, sas=sas_t1)
c_ss_t1 = _steady_state_concentrations_from_dataset(ds_t1)

# Assertions
_assert(set(c_ss_t1.keys()) == {"s1", "s2"}, "keys == {s1, s2}")
np.testing.assert_allclose(c_ss_t1["s1"], 0.5 * 1.0)
_assert(True, "c_SS[s1] == A[0,0] * tau[0] = 0.5")
np.testing.assert_allclose(c_ss_t1["s2"], 0.8 * 0.5)
_assert(True, "c_SS[s2] == A[1,1] * tau[1] = 0.4")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: A-matrix as heatmap
ax = axes[0]
im = ax.imshow(a_matrix_t1, cmap="Blues", vmin=0)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{a_matrix_t1[i, j]:.2f}", ha="center", va="center", fontsize=12)
ax.set_xticks([0, 1])
ax.set_xticklabels(species_t1)
ax.set_yticks([0, 1])
ax.set_yticklabels([f"comp {k + 1}" for k in range(2)])
ax.set_title("A-matrix (diagonal)")
plt.colorbar(im, ax=ax)

# Right: computed c_SS vs expected
ax2 = axes[1]
x = np.arange(len(species_t1))
expected_t1 = np.array([0.5, 0.4])
computed_t1 = np.array([c_ss_t1[sp] for sp in species_t1])
ax2.bar(x - 0.18, expected_t1, width=0.35, label="expected", color="steelblue")
ax2.bar(x + 0.18, computed_t1, width=0.35, label="computed", color="tomato", alpha=0.75)
ax2.set_xticks(x)
ax2.set_xticklabels(species_t1)
ax2.set_ylabel("$c_{SS}$")
ax2.set_title("Steady-state concentrations")
ax2.legend()

plt.suptitle("Test 1 — Simple diagonal A-matrix", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Test 2 — No A-matrix → empty dict

In [ ]:
ds_no_a = xr.Dataset({"species_associated_spectra": xr.DataArray(np.ones((3, 2)))})
result_t2 = _steady_state_concentrations_from_dataset(ds_no_a)

_assert(result_t2 == {}, "returns empty dict when no a_matrix variable present")
print("  (no plot — no data to visualise)")

---
## Test 3 — End-to-end spectra values

With an identity A-matrix and rates $[1, 2]$:
$c_{SS} = [1, 0.5]$, so per-species spectra $= c_{SS,i} \cdot \text{SAS}_i$.

In [ ]:
rng = np.random.default_rng(42)
species_t3 = ["s1", "s2"]
rates_t3 = np.array([1.0, 2.0])
a_matrix_t3 = np.diag([1.0, 1.0])
sas_t3 = rng.random((5, 2))  # random SAS
spectral_t3 = np.linspace(600, 750, 5)

ds_t3 = _make_dataset(
    species=species_t3, rates=rates_t3, a_matrix=a_matrix_t3, sas=sas_t3, n_spectral=5
)
out_t3 = compute_steady_state_spectra(_FakeResult({"ds1": ds_t3}))["ds1"]

c_ss_expected_t3 = np.array([1.0, 0.5])
expected_per_species_t3 = sas_t3 * c_ss_expected_t3[np.newaxis, :]
expected_total_t3 = expected_per_species_t3.sum(axis=1)

np.testing.assert_allclose(out_t3["steady_state_spectra"].values, expected_per_species_t3)
_assert(True, "steady_state_spectra values match c_SS * SAS")
np.testing.assert_allclose(out_t3["steady_state_spectrum"].values, expected_total_t3)
_assert(True, "steady_state_spectrum == sum over species")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

ax = axes[0]
for j, sp in enumerate(species_t3):
    ax.plot(spectral_t3, sas_t3[:, j], label=sp)
ax.set_title("Input SAS")
ax.set_xlabel("Wavelength (nm)")
ax.legend()

ax = axes[1]
for j, sp in enumerate(species_t3):
    ax.plot(spectral_t3, expected_per_species_t3[:, j], label=f"{sp} (expected)", linestyle="--")
    ax.plot(
        spectral_t3,
        out_t3["steady_state_spectra"].sel(species=sp).values,
        label=f"{sp} (computed)",
        linestyle="-",
        alpha=0.6,
    )
ax.set_title("Per-species $e_{steady}$")
ax.set_xlabel("Wavelength (nm)")
ax.legend(fontsize=7)

ax = axes[2]
ax.plot(spectral_t3, expected_total_t3, label="expected total", linestyle="--", linewidth=2)
ax.plot(
    spectral_t3,
    out_t3["steady_state_spectrum"].values,
    label="computed total",
    linestyle="-",
    alpha=0.6,
    linewidth=2,
)
ax.set_title("Total steady-state spectrum")
ax.set_xlabel("Wavelength (nm)")
ax.legend()

plt.suptitle("Test 3 — End-to-end spectra values", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Test 4 — Correct output dimensions

In [ ]:
species_t4 = ["s1", "s2", "s3"]
rates_t4 = np.array([0.5, 1.0, 4.0])
a_matrix_t4 = np.eye(3)
sas_t4 = np.ones((8, 3))

ds_t4 = _make_dataset(
    species=species_t4, rates=rates_t4, a_matrix=a_matrix_t4, sas=sas_t4, n_spectral=8
)
out_t4 = compute_steady_state_spectra(_FakeResult({"d": ds_t4}))["d"]

_assert(
    out_t4["steady_state_spectra"].dims == ("spectral", "species"), "dims == (spectral, species)"
)
_assert(out_t4["steady_state_spectrum"].dims == ("spectral",), "dims == (spectral,)")
_assert(
    list(out_t4["steady_state_spectra"].coords["species"].values) == species_t4,
    "species coord order preserved",
)

print("\nOutput dataset:")
print(out_t4)

# ── Plot: show the dimension layout ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
spectral_t4 = np.linspace(600, 750, 8)
c_ss_t4 = np.array([1.0 / r for r in rates_t4])  # identity A → c_SS = tau
for j, sp in enumerate(species_t4):
    ax.plot(spectral_t4, out_t4["steady_state_spectra"].sel(species=sp).values, label=sp)
ax.plot(spectral_t4, out_t4["steady_state_spectrum"].values, "k--", linewidth=2, label="total")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("$e_{steady}$")
ax.set_title("Test 4 — Correct dimensions (SAS=1 everywhere, so $e_i = c_{SS,i}$)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Test 5 — Skips dataset without A-matrix

In [ ]:
ds_no_a2 = xr.Dataset({"species_associated_spectra": xr.DataArray(np.ones((3, 2)))})
out_t5 = compute_steady_state_spectra(_FakeResult({"no_a": ds_no_a2}))

_assert(out_t5 == {}, "dataset without a_matrix is silently excluded from output")
print("  (no plot — dataset was skipped, nothing to visualise)")

---
## Test 6 — Multi-megacomplex accumulation

Two megacomplexes (`mc1`, `mc2`) each contribute to a different species.
Their contributions to $c_{SS}$ are summed:

$$c_{SS,s1} = 0.6 \cdot \frac{1}{1} = 0.6, \quad c_{SS,s2} = 0.4 \cdot \frac{1}{2} = 0.2$$

In [ ]:
species_t6 = ["s1", "s2"]
spectral_t6 = np.linspace(600, 750, 4)


def _a_da(a, rates, mc_label):
    c_name = f"component_{mc_label}"
    s_name = f"species_{mc_label}"
    return xr.DataArray(
        a,
        dims=(c_name, s_name),
        coords={
            c_name: np.arange(1, rates.size + 1),
            f"lifetime_{mc_label}": (c_name, 1.0 / rates),
            f"rate_{mc_label}": (c_name, rates),
            s_name: species_t6,
        },
    )


sas_t6 = xr.DataArray(
    np.ones((4, 2)),
    dims=("spectral", "species"),
    coords={"spectral": spectral_t6, "species": species_t6},
)
ds_t6 = xr.Dataset(
    {
        "a_matrix_mc1": _a_da(np.array([[0.6, 0.0]]), np.array([1.0]), "mc1"),
        "a_matrix_mc2": _a_da(np.array([[0.0, 0.4]]), np.array([2.0]), "mc2"),
        "species_associated_spectra": sas_t6,
    }
)

out_t6 = compute_steady_state_spectra(_FakeResult({"d": ds_t6}))["d"]

np.testing.assert_allclose(
    out_t6["steady_state_spectra"].sel(species="s1").values, np.full(4, 0.6)
)
_assert(True, "c_SS[s1] == 0.6 (from mc1 only)")
np.testing.assert_allclose(
    out_t6["steady_state_spectra"].sel(species="s2").values, np.full(4, 0.2)
)
_assert(True, "c_SS[s2] == 0.2 (from mc2 only)")
np.testing.assert_allclose(out_t6["steady_state_spectrum"].values, np.full(4, 0.8))
_assert(True, "total == 0.8")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: A-matrices side by side
ax = axes[0]
a1 = np.array([[0.6, 0.0]])
a2 = np.array([[0.0, 0.4]])
x_pos = np.arange(2)
ax.bar(x_pos - 0.2, a1[0], width=0.35, label="mc1 (rate=1)", color="steelblue")
ax.bar(x_pos + 0.2, a2[0], width=0.35, label="mc2 (rate=2)", color="darkorange")
ax.set_xticks(x_pos)
ax.set_xticklabels(species_t6)
ax.set_ylabel("Amplitude $A_{1,i}$")
ax.set_title("A-matrices per megacomplex")
ax.legend()

# Right: resulting steady-state spectra (SAS=1 so value = c_SS)
ax2 = axes[1]
ax2.plot(
    spectral_t6,
    out_t6["steady_state_spectra"].sel(species="s1").values,
    label="s1 (c_SS=0.6)",
    color="steelblue",
)
ax2.plot(
    spectral_t6,
    out_t6["steady_state_spectra"].sel(species="s2").values,
    label="s2 (c_SS=0.2)",
    color="darkorange",
)
ax2.plot(
    spectral_t6,
    out_t6["steady_state_spectrum"].values,
    label="total (0.8)",
    color="black",
    linestyle="--",
    linewidth=2,
)
ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel("$e_{steady}$ (SAS=1 everywhere)")
ax2.set_title("Accumulated steady-state spectra")
ax2.legend()

plt.suptitle("Test 6 — Multi-megacomplex accumulation", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Test 7 — `exclude_species` removes scatter species

A scatter species with a very high amplitude and fast rate is excluded.
The output should contain only `s1`.

In [ ]:
species_t7 = ["s1", "scatter"]
rates_t7 = np.array([1.0, 100.0])
a_matrix_t7 = np.diag([1.0, 5.0])  # scatter has high amplitude
sas_t7 = np.ones((5, 2))
spectral_t7 = np.linspace(600, 750, 5)

ds_t7 = _make_dataset(
    species=species_t7, rates=rates_t7, a_matrix=a_matrix_t7, sas=sas_t7, n_spectral=5
)
out_with_scatter = compute_steady_state_spectra(_FakeResult({"d": ds_t7}))["d"]
out_no_scatter = compute_steady_state_spectra(
    _FakeResult({"d": ds_t7}), exclude_species=["scatter"]
)["d"]

_assert(
    list(out_no_scatter["steady_state_spectra"].coords["species"].values) == ["s1"],
    "scatter species removed from coords",
)
np.testing.assert_allclose(out_no_scatter["steady_state_spectrum"].values, np.ones(5))
_assert(True, "total spectrum == c_SS[s1]*SAS_s1 = 1.0 everywhere")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for sp in species_t7:
    ax.plot(spectral_t7, out_with_scatter["steady_state_spectra"].sel(species=sp).values, label=sp)
ax.plot(
    spectral_t7,
    out_with_scatter["steady_state_spectrum"].values,
    "k--",
    linewidth=2,
    label="total",
)
ax.set_title("Without exclusion\n(scatter dominates)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("$e_{steady}$")
ax.legend()

ax2 = axes[1]
ax2.plot(spectral_t7, out_no_scatter["steady_state_spectra"].sel(species="s1").values, label="s1")
ax2.plot(
    spectral_t7, out_no_scatter["steady_state_spectrum"].values, "k--", linewidth=2, label="total"
)
ax2.set_title("With exclude_species=['scatter']\n(only s1 remains)")
ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel("$e_{steady}$")
ax2.legend()

plt.suptitle("Test 7 — exclude_species removes scatter", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Test 8 — Excluding absent species is a no-op

Passing a species name that does not exist in the dataset should not change the result.

In [ ]:
species_t8 = ["s1", "s2"]
rates_t8 = np.array([1.0, 2.0])
a_matrix_t8 = np.diag([1.0, 1.0])
sas_t8 = np.ones((5, 2))
spectral_t8 = np.linspace(600, 750, 5)

ds_t8 = _make_dataset(
    species=species_t8, rates=rates_t8, a_matrix=a_matrix_t8, sas=sas_t8, n_spectral=5
)
out_filtered = compute_steady_state_spectra(
    _FakeResult({"d": ds_t8}), exclude_species=["nonexistent"]
)["d"]
out_unfiltered = compute_steady_state_spectra(_FakeResult({"d": ds_t8}))["d"]

np.testing.assert_allclose(
    out_filtered["steady_state_spectrum"].values,
    out_unfiltered["steady_state_spectrum"].values,
)
_assert(True, "exclude_species=['nonexistent'] does not change the result")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    spectral_t8,
    out_unfiltered["steady_state_spectrum"].values,
    label="unfiltered",
    linewidth=2,
    color="steelblue",
)
ax.plot(
    spectral_t8,
    out_filtered["steady_state_spectrum"].values,
    label="exclude_species=['nonexistent']",
    linewidth=2,
    linestyle="--",
    color="tomato",
    alpha=0.7,
)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("$e_{steady}$ (total)")
ax.set_title("Test 8 — Excluding absent species is a no-op\n(curves should overlap exactly)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

All 8 tests passed. ✓